# Dissertation dataset overview

This notebook presents repository-safe metadata and a fully synthetic option
sample. It does **not** contain or download licensed OptionMetrics records.

# 论文数据集概览

本 Notebook 仅展示可公开的汇总元数据和完全合成的期权样本，不包含、下载或
复制任何受许可保护的 OptionMetrics 原始记录。

In [ ]:
from pathlib import Path
import csv
import html
import json

try:
    from IPython.display import Markdown, SVG, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    class SVG(str):
        pass

    def display(value):
        print(str(value))


def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples" / "synthetic_option_data.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


ROOT = find_repository_root()
PROFILE_PATH = ROOT / "docs" / "data" / "dataset_profile.json"
SYNTHETIC_PATH = ROOT / "examples" / "synthetic_option_data.csv"

## Aggregate profile / 汇总概况

The profile contains file-level counts, date ranges, missing-value counts, and
selected numeric summaries only. It contains no individual observations.

该概况仅包含文件级行数、日期范围、缺失值数量和部分数值摘要，不包含单条观测。

In [ ]:
if PROFILE_PATH.is_file():
    profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
    lines = [
        "| Table | Rows | Period | Size (MiB) |",
        "|---|---:|---|---:|",
    ]
    for name, table in profile["tables"].items():
        period = f'{table["date_range"]["min"]} to {table["date_range"]["max"]}'
        size_mib = table["file_size_bytes"] / (1024 ** 2)
        lines.append(f'| {name} | {table["row_count"]:,} | {period} | {size_mib:,.2f} |')
    display(Markdown("\n".join(lines)))
else:
    display(Markdown(
        "**Aggregate profile not generated yet.** Run "
        "`python3 scripts/profile_datasets.py --data-dir data "
        "--output docs/data/dataset_profile.json` locally."
    ))

## Synthetic option sample / 合成期权样本

Every row below is invented for demonstration. The `synthetic_only` marker and
2030 dates distinguish it from the dissertation extracts.

以下记录均为演示而人工构造；`synthetic_only` 标志和 2030 年日期用于明确区分
论文使用的真实提取数据。

In [ ]:
with SYNTHETIC_PATH.open("r", encoding="utf-8", newline="") as handle:
    synthetic_rows = list(csv.DictReader(handle))

preview_fields = ["date", "symbol", "cp_flag", "strike_price", "impl_volatility"]
lines = [
    "| Date | Symbol | Type | Strike | Synthetic IV |",
    "|---|---|:---:|---:|---:|",
]
for row in synthetic_rows:
    strike = float(row["strike_price"]) / 1000
    lines.append(
        f'| {row["date"]} | {row["symbol"]} | {row["cp_flag"]} | '
        f'{strike:,.0f} | {float(row["impl_volatility"]):.1%} |'
    )
display(Markdown("\n".join(lines)))

## Illustrative volatility smile / 波动率微笑示意

The chart is generated from the synthetic rows and is not an empirical result.

该图完全由合成记录生成，不属于论文的实证结果。

In [ ]:
points = sorted(
    (
        float(row["strike_price"]) / 1000,
        float(row["impl_volatility"]),
    )
    for row in synthetic_rows
)
width, height = 700, 360
left, right, top, bottom = 70, 25, 35, 55
x_min, x_max = min(x for x, _ in points), max(x for x, _ in points)
y_min, y_max = 0.17, 0.26


def x_pixel(value):
    return left + (value - x_min) / (x_max - x_min) * (width - left - right)


def y_pixel(value):
    return top + (y_max - value) / (y_max - y_min) * (height - top - bottom)


polyline = " ".join(f"{x_pixel(x):.1f},{y_pixel(y):.1f}" for x, y in points)
circles = "".join(
    f'<circle cx="{x_pixel(x):.1f}" cy="{y_pixel(y):.1f}" r="5" fill="#2563eb"/>'
    for x, y in points
)
x_labels = "".join(
    f'<text x="{x_pixel(x):.1f}" y="{height - 20}" text-anchor="middle" '
    f'font-size="12">{x:,.0f}</text>'
    for x, _ in points
)
y_ticks = [0.18, 0.20, 0.22, 0.24, 0.26]
y_labels = "".join(
    f'<text x="{left - 10}" y="{y_pixel(y) + 4:.1f}" text-anchor="end" '
    f'font-size="12">{y:.0%}</text>'
    for y in y_ticks
)
svg = f'''
<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">
  <rect width="100%" height="100%" fill="white"/>
  <line x1="{left}" y1="{top}" x2="{left}" y2="{height-bottom}" stroke="#475569"/>
  <line x1="{left}" y1="{height-bottom}" x2="{width-right}" y2="{height-bottom}" stroke="#475569"/>
  <polyline points="{html.escape(polyline)}" fill="none" stroke="#2563eb" stroke-width="3"/>
  {circles}
  {x_labels}
  {y_labels}
  <text x="{width/2}" y="18" text-anchor="middle" font-size="16">Synthetic volatility smile</text>
  <text x="{width/2}" y="{height-2}" text-anchor="middle" font-size="13">Strike</text>
  <text x="16" y="{height/2}" text-anchor="middle" font-size="13"
        transform="rotate(-90 16 {height/2})">Implied volatility</text>
</svg>
'''
display(SVG(svg))